# Pandas — E-commerce Business Transaction

Para Pandas, el objetivo será realizar el procesamiento y análisis exploratorio del dataset de transacciones de comercio electrónico, aplicando operaciones de limpieza, transformación, filtrado, agrupación, agregación y cálculo de métricas.

Las consultas estarán diseñadas para representar operaciones habituales en un escenario de negocio de e-commerce, como análisis de ventas mensuales, ingresos por país, productos más vendidos y clientes con mayor gasto.

## Instalación de dependencias

In [2]:
!pip install -q kagglehub matplotlib
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Descarga del dataset

In [3]:
import kagglehub

DATASET_DIR = Path.cwd().parent / "dataset"
LOCAL_FILE = DATASET_DIR / "sales_transaction.csv"

if not LOCAL_FILE.exists():
    cached = Path(kagglehub.dataset_download("gabrielramos87/an-online-shop-business"))
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    LOCAL_FILE.write_bytes((cached / "Sales Transaction v.4a.csv").read_bytes())
    print(f"Dataset descargado en: {LOCAL_FILE}")
else:
    print(f"Dataset ya disponible: {LOCAL_FILE}")

Dataset ya disponible: /home/bricafio/Escritorio/BigData/dataset/sales_transaction.csv


## Lectura del Dataset

In [4]:
df = pd.read_csv(LOCAL_FILE,dtype={"TransactionNo": "string","ProductNo": "string","CustomerNo": "string"},na_values=["NA"])

print(f"Dimensiones del dataset: {df.shape}")

Dimensiones del dataset: (536350, 8)


In [5]:
df.head()

,TransactionNo,Date,ProductNo,ProductName,Price,Quantity,CustomerNo,Country
0,581482,12/9/2019,22485,Set Of 2 Wooden Market Crates,21.47,12,17490,United Kingdom
1,581475,12/9/2019,22596,Christmas Star Wish List Chalkboard,10.65,36,13069,United Kingdom
2,581475,12/9/2019,23235,Storage Tin Vintage Leaf,11.53,12,13069,United Kingdom
3,581475,12/9/2019,23272,Tree T-Light Holder Willie Winkie,10.65,12,13069,United Kingdom
4,581475,12/9/2019,23239,Set Of 4 Knick Knack Tins Poppies,11.94,6,13069,United Kingdom


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 536350 entries, 0 to 536349
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   TransactionNo  536350 non-null  string 
 1   Date           536350 non-null  object 
 2   ProductNo      536350 non-null  string 
 3   ProductName    536350 non-null  object 
 4   Price          536350 non-null  float64
 5   Quantity       536350 non-null  int64  
 6   CustomerNo     536295 non-null  string 
 7   Country        536350 non-null  object 
dtypes: float64(1), int64(1), object(3), string(3)
memory usage: 32.7+ MB


## Realizamos una pequeña exploración inicial de los datos

In [7]:
# Variables del df y su tipo de dato.
df.dtypes

TransactionNo    string[python]
Date                     object
ProductNo        string[python]
ProductName              object
Price                   float64
Quantity                  int64
CustomerNo       string[python]
Country                  object
dtype: object

In [8]:
# Vemos el df que contenga al menos un valor nulo en su fila.
nulls = df.isnull().sum().sort_values(ascending=False)
nulls[nulls > 0]

CustomerNo    55
dtype: int64

In [9]:
# También contamos la cantidad de duplicados.
duplicated_count = df.duplicated().sum()

print(f"Registros duplicados: {duplicated_count:,}")

Registros duplicados: 5,200


In [10]:
# Y por último, vemos unas estadisticas iniciales del df.
df[["Price", "Quantity"]].describe()

,Price,Quantity
count,536350.000000,536350.000000
mean,12.662182,9.919347
std,8.490450,216.662300
min,5.130000,-80995.000000
25%,10.990000,1.000000
50%,11.940000,3.000000
75%,14.090000,10.000000
max,660.620000,80995.000000


## Consultas

### Consulta 1 : Identificación y eliminación de valores nulos
Rubro : Limpieza de datos

In [11]:
# Primero contamos los registros.
null_customer = df["CustomerNo"].isna().sum()

print(f"Registros con CustomerNo nulo: {null_customer:,}")

Registros con CustomerNo nulo: 55


In [12]:
# Y despues los eliminamos.
df = df.dropna(subset=["CustomerNo"]).copy()

print(f"Dimensiones después de eliminar nulos: {df.shape}")

Dimensiones después de eliminar nulos: (536295, 8)


### Consulta 2 : Eliminación de duplicados
Rubro : Deduplicación

In [13]:
# Primero contamos los duplicados.
duplicates = df.duplicated().sum()

print(f"Registros duplicados: {duplicates:,}")

Registros duplicados: 5,200


In [14]:
# Y despues los eliminamos, manteniendo la primera aparición.
df = df.drop_duplicates(keep="first").copy()

print(f"Dimensiones después de eliminar duplicados: {df.shape}")

Dimensiones después de eliminar duplicados: (531095, 8)


### Consulta 3 : Transformación de fechas
Rubro: Transformación de variables

In [15]:
# Tratamos a la variable Date como fecha y no como número.
df["Date"] = pd.to_datetime(df["Date"],dayfirst=False,errors="coerce")

In [16]:
# Ahora creamos las variables derivadas.
df["Año"] = df["Date"].dt.year
df["Mes"] = df["Date"].dt.month
df["Día"] = df["Date"].dt.day

In [17]:
df[["Año","Mes","Día"]].head()

,Año,Mes,Día
0,2019,12,9
1,2019,12,9
2,2019,12,9
3,2019,12,9
4,2019,12,9


### Consulta 4 : Creamos TotalSales
Rubro : Transformación de variables

In [18]:
df["TotalSales"] = df["Price"] * df["Quantity"]

In [19]:
# Verificamos
df[["Price","Quantity","TotalSales"]].head()

,Price,Quantity,TotalSales
0,21.47,12,257.64
1,10.65,36,383.40
2,11.53,12,138.36
3,10.65,12,127.80
4,11.94,6,71.64


### Consulta 5 : Filtrado de transacciones válidas
Rubro : Filtrado

In [20]:
valid_transactions = ((df["Quantity"] >= 0) & (~df["TransactionNo"].str.startswith("C", na=False)))

df = df[valid_transactions].copy()

print(f"Dimensiones después del filtrado: {df.shape}")

Dimensiones después del filtrado: (522601, 12)


### Consulta 6 : Facturación mensual
Rubro : Agrupación y agregación

In [21]:
facturacion_mensual = (df.groupby(["Año", "Mes"], as_index=False)["TotalSales"].sum().sort_values(["Año", "Mes"]))

In [22]:
facturacion_mensual

,Año,Mes,TotalSales
0,2018,12,4397648.39
1,2019,1,4548423.47
2,2019,2,3327342.64
3,2019,3,4384669.82
4,2019,4,3579310.06
5,2019,5,4569952.21
6,2019,6,4486050.15
7,2019,7,4571494.88
8,2019,8,4749801.23
9,2019,9,6613772.79


### Consulta 7 : Ingresos por país
Rubro: Agregación

In [23]:
Ingresos_pais = (df.groupby("Country", as_index=False)["TotalSales"].sum().sort_values("TotalSales", ascending=False))

In [24]:
Ingresos_pais.head(10)

,Country,TotalSales
36,United Kingdom,52346795.60
24,Netherlands,2151553.59
10,EIRE,1711819.39
14,Germany,1369839.62
13,France,1329903.39
0,Australia,995414.01
32,Sweden,401879.89
33,Switzerland,361691.96
20,Japan,293155.44
31,Spain,280843.80


### Consulta 8 : Top 10 productos por unidades vendidas
Rubro :  Agrupación y ordenamiento

In [25]:
product_sales = (df.groupby(["ProductNo", "ProductName"],as_index=False)["Quantity"].sum().sort_values("Quantity", ascending=False))

In [26]:
top_10_productos = product_sales.head(10)

top_10_productos

,ProductNo,ProductName,Quantity
2446,23843,Paper Craft Little Birdie,80995
2004,23166,Medium Ceramic Top Storage Jar,78033
1094,22197,Popcorn Holder,56902
2840,84077,World War 2 Gliders Asstd Designs,54951
3256,85099B,Jumbo Bag Red Retrospot,48375
3271,85123A,Cream Hanging Heart T-Light Holder,37937
426,21212,Pack Of 72 Retrospot Cake Cases,36492
3095,84879,Assorted Colour Bird Ornament,36394
1926,23084,Rabbit Night Light,30742
1359,22492,Mini Paint Set Vintage,26633


### Consulta 9 : Top 10 clientes por gastos
Rubro : Agrupación, agregación y ordenamiento

In [27]:
customer_sales = (df.groupby("CustomerNo", as_index=False)["TotalSales"].sum().sort_values("TotalSales", ascending=False))

In [28]:
top_10_customers = customer_sales.head(10)

top_10_customers

,CustomerNo,TotalSales
1880,14646,2112282.03
3302,16446,1002741.57
2085,14911,914204.19
126,12415,900545.54
4581,18102,897137.36
4082,17450,891069.53
68,12346,840113.80
1506,14156,694202.51
1140,13694,646116.78
4127,17511,639006.19


### Consulta 10 : Estadísticos descriptivos
Rubro : Cálculo de métricas

In [29]:
numeric_stats = df[["Price", "Quantity", "TotalSales"]].agg([
    "mean",
    "median",
    "min",
    "max",
    "std"
])

In [30]:
numeric_stats

,Price,Quantity,TotalSales
mean,12.637160,10.667492,1.201324e+02
median,11.940000,4.000000,4.448000e+01
min,5.130000,1.000000,5.130000e+00
max,660.620000,80995.000000,1.002718e+06
std,7.965974,157.542420,1.860159e+03


### Consulta 11 : Tickets promedio por línea
Rubro : Cálculo de métricas

In [31]:
ventas_promedio = df["TotalSales"].mean()

print(f"Venta promedio por línea: £{ventas_promedio:,.2f}")

Venta promedio por línea: £120.13


### Consulta 12 : Número de productos únicos
Rubro : Cálculo de métricas

In [32]:
productos_unicos = df["ProductNo"].nunique()

print(f"Productos únicos: {productos_unicos:,}")

Productos únicos: 3,753


### Consulta 13 : Ventas por día de la semana
Rubro : Transformación, agrupamiento, agregación y ordenamiento.

In [33]:
df["DiaSemana"] = df["Date"].dt.day_name()

In [34]:
ventas_semana = (df.groupby("DiaSemana", as_index=False)["TotalSales"].sum())

In [35]:
orden_semana = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

ventas_semana["DiaSemana"] = pd.Categorical(ventas_semana["DiaSemana"],categories=orden_semana,ordered=True)

ventas_semana = ventas_semana.sort_values("DiaSemana")

In [36]:
ventas_semana

,DiaSemana,TotalSales
1,Monday,10285588.97
5,Wednesday,5280327.03
4,Thursday,9857250.51
0,Friday,12678747.95
2,Saturday,11193999.01
3,Sunday,13485391.07
